# LFW·SurvFace 공통 Quick/Full 실험 실행기

이 노트북은 데이터셋별 legacy 노트북을 삭제하거나 서로 실행하지 않습니다. 동일한 `research/` Python 단계 함수를 한 곳에서 계획하고 순차 호출하는 상위 실행기입니다.

- `quick`: 실제 데이터와 실제 모델을 사용하되 LFW 10%, SurvFace 2%의 identity-aware·role-preserving 표본을 사용합니다.
- `full`: 전체 데이터 100%를 사용합니다.
- 실제 장시간 실험은 사용자가 직접 `EXECUTE=True`와 실행 확인 값을 설정해야 시작됩니다.
- 현재 공통 dispatcher는 기존 Step 4 Grad-CAM workflow를 지원합니다. 새 평가 계약의 exhaustive ADC, IVF-PQ, calibration 100/500/1,000 행렬은 아직 구현 전이므로 이 노트북의 full 완료만으로 논문 최종 비교가 되지는 않습니다.


## 1. 사용자가 조절하는 변수

`DATASET_ID`와 `RUN_TIER`를 선택합니다. fraction은 연구 계약으로 고정되어 있으므로 여기서 임의로 입력하지 않습니다. `START_NEW_RUN=True`는 동일 plan의 완료 run이 이미 있는데도 독립 재실험을 의도할 때만 사용합니다.


In [ ]:
from __future__ import annotations

DATASET_ID = "survface"  # "lfw" 또는 "survface"
RUN_TIER = "quick"       # "quick" 또는 "full"
SEED = 42
MODEL_PROFILE = None      # None이면 추적된 Step 4 config의 등록 모델 사용

EXECUTE = False
ACKNOWLEDGE_LOCAL_EXECUTION = False
START_NEW_RUN = False


## 2. 프로젝트와 공통 runner 로드

현재 작업 디렉터리의 상위에서 저장소 루트를 찾습니다. 아래 셀은 실험을 시작하지 않습니다.


In [ ]:
from pathlib import Path
from pprint import pprint
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.pipeline_runner import (
    FULL_DATA_FRACTION,
    QUICK_DATA_FRACTIONS,
    build_common_experiment_plan,
    inspect_common_experiment_plan,
    run_common_step4_experiment,
)
from research.runtime import ProgressReporter

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"QUICK_DATA_FRACTIONS={dict(QUICK_DATA_FRACTIONS)}")
print(f"FULL_DATA_FRACTION={FULL_DATA_FRACTION}")


## 3. 결정적 실행 plan 생성

manifest를 읽어 실제 선택 예정 행 수와 role/split 분포를 계산합니다. 같은 manifest hash·seed·tier라면 같은 identity 집합을 선택합니다. 이 셀도 DB나 run을 변경하지 않습니다.


In [ ]:
PLAN = build_common_experiment_plan(
    project_root=PROJECT_ROOT,
    dataset_id=DATASET_ID,
    run_tier=RUN_TIER,
    seed=SEED,
    model_profile=MODEL_PROFILE,
)
pprint(PLAN.as_dict(), sort_dicts=False)


## 4. 로컬 preflight

등록 checkpoint, CUDA, ONNX Runtime provider, canonical aligned/landmark bundle, Git source 상태를 읽기 전용으로 검사합니다. GitHub나 원격 CI는 사용하지 않습니다. `ready_to_execute_pipeline=False`이면 아래 실행 셀을 켜지 말고 `readiness.checks`의 실패 항목을 먼저 해결합니다.


In [ ]:
PREFLIGHT = inspect_common_experiment_plan(PLAN)
pprint(PREFLIGHT, sort_dicts=False)


## 5. 사용자 승인 후 순차 실행 또는 재개

기본값에서는 실행하지 않습니다. 실제 실행 시 첫 셀에서 `EXECUTE=True`, `ACKNOWLEDGE_LOCAL_EXECUTION=True`로 바꾼 뒤 커널을 재시작하고 전체 실행합니다.

완료된 phase는 건너뛰고, 실패하거나 아직 실행하지 않은 phase부터 이어갑니다. 장시간 loop 로그는 약 10% 경계에서만 출력됩니다. 같은 plan의 완료 run이 있으면 자동으로 새 run을 만들지 않습니다.


In [ ]:
if EXECUTE:
    if ACKNOWLEDGE_LOCAL_EXECUTION is not True:
        raise RuntimeError(
            "실제 실행 전 ACKNOWLEDGE_LOCAL_EXECUTION=True가 필요합니다."
        )
    if not PREFLIGHT["ready_to_execute_pipeline"]:
        raise RuntimeError("preflight 실패 항목을 먼저 해결하십시오.")
    PROGRESS = ProgressReporter(
        f"{DATASET_ID}/{RUN_TIER}",
        heartbeat_seconds=None,
        milestone_percent=10,
    )
    EXECUTION_RESULT = run_common_step4_experiment(
        PLAN,
        execution_acknowledged=True,
        start_new_run=START_NEW_RUN,
        progress=PROGRESS.callback(key_prefix=f"{DATASET_ID}:{RUN_TIER}:"),
    )
else:
    EXECUTION_RESULT = {
        "status": "not_started",
        "reason": "EXECUTE=False; plan과 preflight만 수행했습니다.",
    }

pprint(EXECUTION_RESULT, sort_dicts=False)


## 6. 결과 해석 경계

- `quick` 결과는 코드·DB·artifact 흐름과 경향 확인용이며 논문 최종 수치가 아닙니다.
- `full`은 전체 표본이라는 조건만 충족합니다. 동일 commit에서 LFW/SurvFace를 모두 재실행하고, 모델 UID·전처리·평가 계약 parity를 확인해야 직접 비교할 수 있습니다.
- 현재 runner의 PQ 결과는 기존 reconstruction characterization입니다. exhaustive ADC와 IVF-PQ가 구현되기 전에는 PQ code-search 성능으로 주장하지 않습니다.
- Grad-CAM은 faithfulness, 교란 통제, 독립 표본 추가 예측력, 고정 FPIR 운영 개선, 비용 검증을 통과하기 전까지 보조 설명 결과입니다.
- 데이터셋별 세부 단계 디버깅은 기존 `notebooks/lfw` 또는 `notebooks/survface` 순차 노트북을 사용합니다.
